In [ ]:

import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
#load qwen2.5 model with 0.5 billion parameters
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype
).to(device)

model.eval()

SYSTEM_PROMPT = {
    "role": "system",
    "content": "You are a concise, helpful assistant for NIC e-Governance demos."
}


def ask_bot(message, history):

    # Start with system instructions
    messages = [SYSTEM_PROMPT.copy()]

    # Process previous conversation
    for item in (history or []):

        # New Gradio messages format
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")

            if role in ["user", "assistant"] and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

        # Older Gradio tuple format
        elif isinstance(item, (list, tuple)) and len(item) == 2:

            user_text, assistant_text = item

            if user_text:
                messages.append({
                    "role": "user",
                    "content": user_text
                })

            if assistant_text:
                messages.append({
                    "role": "assistant",
                    "content": assistant_text
                })

    # Add latest message
    messages.append({
        "role": "user",
        "content": message
    })

    # Format conversation for Qwen
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert text into tensors
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(device)

    # Generate answer
    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    generated_ids = output_ids[0][
        inputs["input_ids"].shape[1]:
    ]

    # Decode tokens into readable text
    reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return reply


print("Chatbot ready!")



In [ ]:
demo = gr.ChatInterface(
    fn=ask_bot,
    title="NIC e-Governance AI Assistant",
    description="Powered by Qwen 2.5 0.5B Instruct",
     examples=[
        "Hello! What can you help me with?",
        "Give me a short list of e-governance projects by NIC",
        "Give 5 trending project ideas in Python"
    ]
)
demo.launch()